# Exp 23 — Caption Nity clips with Qwen3-VL-30B (vLLM / Colab GPU)

GPU port of `phase2/exp21_caption_clips.py`. The local MacBook run used Qwen2-VL-**2B** (4-bit), which was too weak — it called the octopus a "fish". This notebook runs the much stronger **Qwen3-VL-30B-A3B-Instruct (AWQ Int4)** MoE on a Colab GPU.

**Why vLLM (not plain transformers):** loading the AWQ 30B through `transformers` 5.x routes to the gptqmodel **Marlin** kernel, which requires every quantized layer's `out_features` to be divisible by 64 — and this model has a 4304-wide layer (4304 ÷ 64 = 67.25), so it crashes at load. **vLLM** has its own AWQ kernels that pad `out_features`, so the same checkpoint loads cleanly and fits on a single **A100-40GB** (~20 GB used).

**For each clip in `data/octopus_clips/`:**
1. Sample frames across the whole 20s clip (`FRAME_FPS`, default 1 frame / 2s ≈ 10 frames)
2. Feed them to Qwen3-VL as an ordered image list and ask for **one caption for the whole clip** — or `not present` if the octopus isn't visible in any frame
3. Map the caption to one ethogram label and save to `captions-<model>.json` (same schema as exp21, plus `n_frames`)

Resumable — skips clips already in `captions.json`.

> **Runtime → Change runtime type → A100 GPU.** First model load downloads ~17 GB. If a cell errors with a torch/CUDA mismatch right after install, do **Runtime → Restart session** and re-run (vLLM may have swapped the torch build).

## 1. Install dependencies

In [ ]:
# ── Why the previous attempt failed ────────────────────────────────────────────
# `pip install -U vllm` pulled vLLM 0.23.0, which is built against CUDA 13 (it needs
# libcudart.so.13). But Colab preinstalls torch 2.11.0+cu128 (CUDA 12.8), which only
# ships libcudart.so.12. Since that torch already satisfied vLLM's `torch` pin, pip
# did NOT swap it — so vLLM's compiled ops can't find the CUDA-13 runtime and
# `from vllm import LLM` fails with "libcudart.so.13: cannot open shared object file".
#
# Fix: remove Colab's cu128 torch FIRST, so vLLM installs the exact torch it was built
# against — that torch bundles the matching nvidia-cuda-*-cu13 libs (incl. libcudart.so.13).
!pip uninstall -y -q torch torchvision torchaudio torchao 2>/dev/null
!pip install -q -U vllm qwen-vl-utils
!apt-get -qq install -y ffmpeg >/dev/null

# Do NOT import vllm here — it only links its CUDA libs on first `from vllm import LLM`,
# and that must happen in a FRESH kernel.
print("✅ Installed.")
print("➡️  NOW: Runtime → Restart session, then run from the CONFIG cell (skip this install cell).")
print("    vLLM cannot be imported in the same kernel that just reinstalled torch.")
print()
print("If after restart the LOAD cell instead says 'CUDA driver version is insufficient',")
print("the Colab GPU driver is older than CUDA 13 — in that case pin a cu128 vLLM:")
print("    !pip install -q 'vllm==0.11.2' qwen-vl-utils   (then Restart session again)")

In [1]:
import torch
print(torch.cuda.get_device_name(0))
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB")
# print("device_map:", getattr(model, "hf_device_map", "n/a"))

NVIDIA A100-SXM4-40GB
VRAM: 42 GB


## 2. Config

In [10]:
# Model — the AWQ Int4 build of the 30B-A3B MoE (~17 GB download, ~20 GB on GPU).
# Served via vLLM, which pads quantized layer dims and runs fine on an A100-40GB.
MODEL        = "QuantTrio/Qwen3-VL-30B-A3B-Instruct-AWQ"
#   "Qwen/Qwen3-VL-30B-A3B-Instruct"   - bf16, ~60 GB download (A100-80GB / H100 only)
#   "Qwen/Qwen3-VL-8B-Instruct"        - dense bf16, ~18 GB, fits anything (lower quality)

FRAME_FPS    = 0.5     # frames per second -> 0.5 = 1 frame every 2s = ~10 frames per 20s clip
MAX_TOKENS   = 200

# vLLM runtime knobs (used by the load cell):
MAX_MODEL_LEN = 8192   # ctx budget: prompt + ~10 image-token blocks + caption
GPU_MEM_UTIL  = 0.92   # fraction of the 40 GB A100 vLLM may reserve
IMAGE_LIMIT   = 16     # max images per prompt (>= frames we ever pass)

from pathlib import Path
# Clips unzip into ./octopus_clips/ on the Colab runtime. This is a LOCAL runtime
# path, NOT a Drive path — do not point it at /content/drive or /home.
CLIPS_DIR     = Path("octopus_clips")
ETHOGRAM_PATH = Path("ethogram_list.json")
MODEL_TAG     = MODEL.split("/")[-1]                     # captions kept per-model
CAPTIONS_PATH = CLIPS_DIR / f"captions-2-{MODEL_TAG}.json"

## 3. Get the data into Colab

Your `octopus_clips.zip` + `ethogram_list.json` are in **`MyDrive/GSOC-Catrobat/`**.

➡️ **Run the FIRST cell below (Option B — Google Drive).** It mounts Drive and unzips the clips — no upload needed.

**Do NOT run the second cell (Option A — Zip upload).** That one opens a file picker and is only for when your files aren't in Drive.

In [3]:
# === Option B: Google Drive  ←←← RUN THIS ONE (no upload needed) ===
# Your zip + json are in MyDrive/GSOC-Catrobat/.
from google.colab import drive
drive.mount("/content/drive")

import zipfile, shutil
DRIVE_ROOT = Path("/content/drive/MyDrive/GSOC-Catrobat")   # your Drive folder

with zipfile.ZipFile(DRIVE_ROOT / "octopus_clips.zip") as z:
    z.extractall(".")                       # creates ./octopus_clips/
shutil.copy(DRIVE_ROOT / "ethogram_list.json", ETHOGRAM_PATH)

if not CLIPS_DIR.exists():                   # zip flattened the mp4s
    CLIPS_DIR.mkdir(parents=True, exist_ok=True)
    for mp4 in Path(".").glob("*.mp4"):
        shutil.move(str(mp4), CLIPS_DIR / mp4.name)

print(len(list(CLIPS_DIR.glob('*.mp4'))), "clips ready")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
52 clips ready


In [4]:
# # === Option A: Zip upload  (SKIP if you ran the Drive cell above) ===
# # Only run this if your files are NOT in Drive. It opens a file picker.
# import zipfile, shutil
# from google.colab import files

# up = files.upload()   # select octopus_clips.zip and ethogram_list.json

# for name in up:
#     if name.endswith(".zip"):
#         with zipfile.ZipFile(name) as z:
#             z.extractall(".")        # zip contains the octopus_clips/ folder
#     elif name == "ethogram_list.json":
#         shutil.move(name, ETHOGRAM_PATH)

# if not CLIPS_DIR.exists():            # zip flattened the mp4s
#     CLIPS_DIR.mkdir(parents=True, exist_ok=True)
#     for mp4 in Path(".").glob("*.mp4"):
#         shutil.move(str(mp4), CLIPS_DIR / mp4.name)

# assert ETHOGRAM_PATH.exists(), "ethogram_list.json not found — upload it too"
# print(len(list(CLIPS_DIR.glob('*.mp4'))), "clips ready")

## 4. Load the model

In [5]:
from vllm import LLM, SamplingParams
from transformers import AutoProcessor

# vLLM auto-detects AWQ from the checkpoint config and uses its awq_marlin kernel,
# which PADS out_features — so the 30B-A3B's 4304-wide layer loads cleanly (the
# exact thing that crashed the transformers/gptqmodel path).
print(f"Loading {MODEL} with vLLM …  (first run downloads ~17 GB)")
llm = LLM(
    model=MODEL,
    max_model_len=MAX_MODEL_LEN,
    gpu_memory_utilization=GPU_MEM_UTIL,
    limit_mm_per_prompt={"image": IMAGE_LIMIT},
    dtype="auto",
    trust_remote_code=True,
)

# The processor is only used to build the chat-template string + preprocess frames;
# generation happens in vLLM.
processor = AutoProcessor.from_pretrained(MODEL, trust_remote_code=True)
sampling_params = SamplingParams(temperature=0.0, max_tokens=MAX_TOKENS)
print("Model ready (vLLM).")

Loading QuantTrio/Qwen3-VL-30B-A3B-Instruct-AWQ with vLLM …  (first run downloads ~17 GB)
INFO 06-22 19:24:28 [api_utils.py:273] non-default args: {'trust_remote_code': True, 'max_model_len': 8192, 'disable_log_stats': True, 'limit_mm_per_prompt': {'image': 16}, 'model': 'QuantTrio/Qwen3-VL-30B-A3B-Instruct-AWQ'}


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:134: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/1.96k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

INFO 06-22 19:24:54 [model.py:611] Resolved architecture: Qwen3VLMoeForConditionalGeneration
INFO 06-22 19:24:54 [model.py:1745] Using max model len 8192
INFO 06-22 19:24:55 [awq_marlin.py:269] The model is convertible to awq_marlin during runtime. Using awq_marlin kernel.
INFO 06-22 19:24:55 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 06-22 19:24:55 [vllm.py:999] Asynchronous scheduling is enabled.
INFO 06-22 19:24:55 [kernel.py:270] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])


...

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/5.50k [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

[transformers] The `use_fast` parameter is deprecated and will be removed in a future version. Use `backend="torchvision"` instead of `use_fast=True`, or `backend="pil"` instead of `use_fast=False`.


WARNING 06-22 19:25:16 [system_utils.py:157] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
Model ready (vLLM).


## 5. Prompt + ethogram matching

The prompt asks for **one caption covering the whole clip** (frames are passed in order as a video), or `not present` if the octopus isn't visible in any frame. The 7B model can follow the structured `CAPTION:` / `ETHOGRAM:` format that the 2B ignored; `match_ethogram()` stays as a keyword fallback, and `parse_response()` normalizes both fields to `not present` when the octopus is absent.

In [6]:
import json

ethogram  = json.load(open(ETHOGRAM_PATH))
behaviors = ethogram["behaviors"]
labels    = [b["label"] for b in behaviors]
valid_set = set(labels)

label_block = "\n".join(f"- {b['label']}: {b['description']}" for b in behaviors)

def build_prompt() -> str:
    return (
        "These frames are sampled in order from a single short aquarium security-camera clip. "
        "The subject is Nity, an octopus (Octopus vulgaris). "
        "Octopuses change color, extend arms, hide in dens, manipulate objects, and interact with humans.\n\n"
        "First decide whether the octopus appears in ANY frame of the clip.\n"
        "- If the octopus is NOT visible in any frame, respond with EXACTLY:\n"
        "  CAPTION: not present\n"
        "  ETHOGRAM: not present\n"
        "- If the octopus IS visible, write ONE caption describing what it does across the whole clip "
        "(its movement, posture, arm position, color, and anything it touches or interacts with), "
        "then pick the single best-matching behavior from this ethogram:\n"
        f"{label_block}\n\n"
        "Respond in EXACTLY this format, nothing else:\n"
        "CAPTION: <one sentence for the whole clip>\n"
        "ETHOGRAM: <one label copied verbatim from the list above>"
    )

# Keyword fallback (same mapping as exp21)
_ETHOGRAM_KEYWORDS = [
    (["crawl", "walking on arms", "moving across"],             "Crawling"),
    (["swim", "jet", "propel", "water column"],                 "Swimming / jetting"),
    (["arm walk", "two arm", "bipedal"],                        "Arm walking"),
    (["hunt", "stalk", "pursuit", "chasing"],                   "Hunting"),
    (["captur", "pounce", "grab", "catch", "seiz"],             "Capturing prey"),
    (["eat", "feeding", "consuming", "tearing food", "food"],   "Manipulating food"),
    (["entering den", "into den", "into shelter", "retreating into"], "Entering den"),
    (["exiting den", "emerging", "leaving den", "coming out"],  "Exiting den"),
    (["rearrang", "piling", "moving shells", "moving rocks", "den entrance"], "Rearranging den"),
    (["extend", "probing", "reaching out", "arm out", "tentacle out"], "Arm extension / probing"),
    (["manipulat", "picking up", "holding object", "playing with"], "Object manipulation"),
    (["above water", "out of water", "water surface"],          "Reaching out of water"),
    (["human", "person", "hand", "researcher", "respond"],      "Responding to human"),
    (["joystick", "toy", "enrichment", "device", "screen"],     "Enrichment interaction"),
    (["color", "colour", "blanch", "darken", "chromatophore", "texture", "camouflage"], "Color / texture change"),
    (["ink", "cloud"],                                          "Ink release"),
    (["hid", "flatten", "conceal", "press"],                    "Hiding / flattening"),
    (["stationary", "resting", "motionless", "still", "not moving", "sitting in den", "inside den"], "Stationary in den"),
    (["stationary", "resting", "motionless", "still", "not moving", "open area", "tank floor"], "Stationary in open"),
]

def match_ethogram(text: str) -> str:
    t = text.lower()
    for keywords, label in _ETHOGRAM_KEYWORDS:
        if any(k in t for k in keywords):
            return label
    return "unknown"

def parse_response(text: str):
    """Pull CAPTION/ETHOGRAM out of the structured response; handle 'not present'."""
    caption, etho = "", None
    for line in text.splitlines():
        s = line.strip()
        if s.upper().startswith("CAPTION:"):
            caption = s[len("CAPTION:"):].strip().strip("'\"")
        elif s.upper().startswith("ETHOGRAM:"):
            raw = s[len("ETHOGRAM:"):].strip().strip("'\"")
            rl = raw.lower()
            for label in valid_set:
                if label.lower() == rl or label.lower() in rl or rl in label.lower():
                    etho = label; break
            else:
                etho = raw if raw else None
    if not caption:
        caption = text.strip().strip("'\"")
    # Octopus absent -> normalize both fields to "not present"
    if "not present" in caption.lower() or "not visible" in caption.lower():
        return "not present", "not present"
    if not etho or etho.lower() in ("unknown", "not present", ""):
        etho = match_ethogram(caption)
    return caption, etho

print(f"{len(labels)} ethogram labels loaded.")

19 ethogram labels loaded.


## 6. Frame sampling + whole-clip inference helpers

In [7]:
import subprocess, tempfile
from qwen_vl_utils import process_vision_info

# Cap each frame's resolution so ~10 frames don't blow up the token count / VRAM.
MAX_PIXELS = 512 * 512

def extract_frames(clip_path: Path, tmpdir: str) -> tuple[list[str], float]:
    """Sample frames evenly across the whole clip at FRAME_FPS. Returns (paths, fps)."""
    pattern = str(Path(tmpdir) / "f_%03d.jpg")
    subprocess.run(
        ["ffmpeg", "-y", "-loglevel", "error",
         "-i", str(clip_path),
         "-vf", f"fps={FRAME_FPS}", "-q:v", "2", pattern],
        check=True,
    )
    frames = sorted(str(p) for p in Path(tmpdir).glob("f_*.jpg"))
    return frames, FRAME_FPS

def caption_clip(frame_paths: list[str], fps: float, prompt: str) -> str:
    """Pass the sampled frames as an ordered image list to vLLM and return the text."""
    content = [{"type": "image", "image": p, "max_pixels": MAX_PIXELS} for p in frame_paths]
    content.append({"type": "text", "text": prompt})
    messages = [{"role": "user", "content": content}]

    # Build the prompt string + decode the frames to PIL via qwen-vl-utils, then
    # hand both to vLLM as one multimodal request.
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, _ = process_vision_info(messages)

    out = llm.generate(
        {"prompt": text, "multi_modal_data": {"image": image_inputs}},
        sampling_params=sampling_params,
        use_tqdm=False,
    )
    return out[0].outputs[0].text.strip()

## 7. Run captioning (resumable)

In [11]:
from datetime import datetime

prompt = build_prompt()
results = json.load(open(CAPTIONS_PATH)) if CAPTIONS_PATH.exists() else []
done = {r["file"] for r in results}

clips = sorted(CLIPS_DIR.glob("*.mp4"))
todo  = [c for c in clips if c.name not in done]
print(f"{len(clips)} clips total, {len(todo)} to caption\n" + "-" * 60)

for i, clip in enumerate(todo):
    print(f"[{i+1}/{len(todo)}] {clip.name}", flush=True)
    with tempfile.TemporaryDirectory() as tmpdir:
        try:
            frames, fps = extract_frames(clip, tmpdir)
            if not frames:
                print("  no frames extracted"); continue
        except Exception as e:
            print(f"  frame extraction failed: {e}"); continue
        try:
            raw = caption_clip(frames, fps, prompt)
        except Exception as e:
            print(f"  inference failed: {e}"); continue

    caption, etho = parse_response(raw)
    print(f"  frames   : {len(frames)}")
    print(f"  caption  : {caption}")
    print(f"  ethogram : {etho}", flush=True)

    results.append({
        "file":         clip.name,
        "caption":      caption,
        "ethogram":     etho,
        "n_frames":     len(frames),
        "raw_response": raw,
        "captioned_at": datetime.now().isoformat(timespec="seconds"),
    })
    with open(CAPTIONS_PATH, "w") as f:
        json.dump(results, f, indent=2)

print("-" * 60 + f"\nDone. {len(results)} captions saved -> {CAPTIONS_PATH}")

from collections import Counter
print("\nEthogram distribution:")
for label, n in Counter(r["ethogram"] for r in results).most_common():
    print(f"  {n:3d}  {label}")

52 clips total, 52 to caption
------------------------------------------------------------
[1/52] 2025-10-28_1514_(?)_w1_Right_Back_00:02-00:22.mp4
  frames   : 10
  caption  : The octopus extends its arms above the water surface, reaching out towards a person who is holding a tool above the tank.
  ethogram : Reaching out of water
[2/52] 2025-10-28_1514_(?)_w2_Right_Back_01:22-01:42.mp4
  frames   : 10
  caption  : The octopus extends its arms above the water surface, reaching out towards a person who is standing nearby, with its body and arms visible through the aquarium glass.
  ethogram : Reaching out of water
[3/52] 2025-11-02_1636_w1_Right_Back_00:01-00:21.mp4
  frames   : 10
  caption  : The octopus is visible on the screen of a television monitor, which is displaying a video of the octopus in a tank. The octopus on the screen changes color and texture, transitioning from a mottled pattern to a lighter, more uniform appearance.
  ethogram : Color / texture change
[4/52] 2025-11-

## 8. Save results back

If you used Drive (Option A), this copies `captions.json` back so it persists. Otherwise it triggers a browser download — drop the file into `data/octopus_clips/captions.json` in the repo.

In [ ]:
import shutil
if Path("/content/drive/MyDrive").exists():
    shutil.copy(CAPTIONS_PATH, DRIVE_ROOT / CAPTIONS_PATH.name)
    print(f"Copied captions back to Drive: {DRIVE_ROOT / CAPTIONS_PATH.name}")
else:
    from google.colab import files
    files.download(str(CAPTIONS_PATH))

Copied captions back to Drive: /content/drive/MyDrive/GSOC-Catrobat/captions-2-Qwen3-VL-30B-A3B-Instruct-AWQ.json


WARNING 06-22 19:42:14 [core_client.py:699] [shutdown] MPClient: engine core exited unexpectedly; starting cleanup
INFO 06-22 19:42:14 [core_client.py:652] [shutdown] MPClient: start timeout=default
INFO 06-22 19:42:14 [core_client.py:654] [shutdown] MPClient: stopping engine manager
INFO 06-22 19:42:14 [core_client.py:656] [shutdown] MPClient: engine manager stopped
INFO 06-22 19:42:14 [core_client.py:657] [shutdown] MPClient: cleaning up background resources
INFO 06-22 19:42:14 [core_client.py:659] [shutdown] MPClient: complete
